# Pandas e Manipulação de Dados — Grupo 2
**ISA - LIA | Nivelamento**

Por: Albertina e Mayara

Neste notebook vamos aprender a usar a biblioteca **Pandas** do Python com um exemplo real: cerca de **10 mil imóveis para alugar** em 5 cidades brasileiras (São Paulo, Rio de Janeiro, Belo Horizonte, Porto Alegre e Campinas).

**Pergunta que queremos responder:** qual cidade tem o aluguel mais caro, e o que faz um imóvel ser mais caro?

**Conteúdo:**
1. Introdução ao Pandas e Estruturas de Dados
2. Importação e Exploração de Dados
3. Seleção e Filtragem de Dados
4. Manipulação de DataFrames

**Fonte dos dados:** OpenML, dataset *Brazilian_houses* (ID 42688), cópia da 2ª versão do dataset "Brazilian houses to rent", originalmente publicado no Kaggle.

## 1. Apresentação
<!-- Mayara: revisar e reescrever com suas palavras -->

Vamos usar um dataset com cerca de **10 mil imóveis para alugar** em 5 cidades brasileiras: São Paulo, Rio de Janeiro, Belo Horizonte, Porto Alegre e Campinas.

Para cada imóvel temos: área, número de quartos, banheiros, vagas de garagem, andar, se aceita animais, se é mobiliado e os valores (aluguel, condomínio, IPTU, seguro incêndio e total).

**Pergunta que queremos responder:** qual cidade tem o aluguel mais caro, e o que faz um imóvel ser mais caro?

**Etapas do notebook:**
1. Apresentação
2. Carregar os dados
3. Conhecer as colunas
4. Limpar os dados
5. Filtrar
6. Agrupar
7. Criar uma coluna nova (preço por m²)
8. Conclusão

**Alguns termos:**
- **Dataset:** um conjunto de dados, geralmente uma tabela (como uma planilha do Excel).
- **Pandas:** biblioteca do Python feita para trabalhar com tabelas.
- **DataFrame:** é como o Pandas chama uma tabela. Vamos guardar a nossa numa variável chamada `df`.
- **Linha:** um imóvel. **Coluna:** uma característica dos imóveis.

**Fonte dos dados:** OpenML, dataset *Brazilian_houses* (ID 42688), cópia da 2ª versão do dataset "Brazilian houses to rent" publicado no Kaggle.

---
# 1. Introdução ao Pandas e Estruturas de Dados
<!-- Mayara: revisar e reescrever com suas palavras -->

## 1.1 O que é Pandas e por que usar
O **Pandas** é a biblioteca do Python mais usada para trabalhar com **dados em tabela**, como uma planilha do Excel. Com ele dá para carregar, explorar, filtrar, organizar e resumir milhares de linhas com poucos comandos.

Em projetos de IA, quase sempre o primeiro passo é abrir os dados com o Pandas para entender o que tem neles.

Para usar, importamos a biblioteca. O `as pd` é um apelido curto, para não precisar escrever `pandas` toda vez.

In [36]:
import pandas as pd

## 1.2 Series
Uma **Series** é uma **coluna** de dados: uma lista de valores, cada um com um rótulo (o **índice**).

In [37]:
alugueis = pd.Series([1500, 2300, 980], index=["apto A", "apto B", "apto C"])
alugueis

,0
apto A,1500
apto B,2300
apto C,980


In [38]:
alugueis["apto B"]

np.int64(2300)

## 1.3 DataFrame
Um **DataFrame** é uma **tabela**: várias Series lado a lado, todas com o mesmo índice. Cada coluna é uma Series.

Podemos criar um DataFrame a partir de um dicionário, onde cada chave vira o nome de uma coluna:

In [39]:
exemplo = pd.DataFrame({
    "cidade": ["Porto Alegre", "Campinas", "São Paulo"],
    "quartos": [2, 1, 3],
    "aluguel": [1500, 980, 3200],
})
exemplo

,cidade,quartos,aluguel
0,Porto Alegre,2,1500
1,Campinas,1,980
2,São Paulo,3,3200


## 1.4 Colunas e índices
- `columns` mostra os nomes das colunas.
- `index` mostra os rótulos das linhas. Quando não dizemos nada, o Pandas numera as linhas a partir de 0.

In [40]:
exemplo.columns

Index(['cidade', 'quartos', 'aluguel'], dtype='object')

In [41]:
exemplo.index

RangeIndex(start=0, stop=3, step=1)

Pegar uma coluna de um DataFrame devolve uma Series:

In [42]:
coluna_aluguel = exemplo["aluguel"]
type(coluna_aluguel)

pandas.core.series.Series

## 1.5 Tipos de dados
Cada coluna tem um **tipo**. Os mais comuns:

| Tipo | O que guarda | Exemplo |
|---|---|---|
| `int64` | números inteiros | quartos |
| `float64` | números com casas decimais | preço por m² |
| `object` | texto | cidade |
| `bool` | Verdadeiro / Falso | resultado de um filtro |

`dtypes` mostra o tipo de cada coluna:

In [43]:
exemplo.dtypes

,0
cidade,object
quartos,int64
aluguel,int64


---
# 2. Importação e Exploração de Dados
<!-- Mayara: revisar e reescrever com suas palavras -->

## 2.1 Importação de arquivos CSV
**CSV** é um arquivo de texto em que cada linha é um registro e os valores são separados por vírgula. É o formato mais comum para compartilhar tabelas.

Nossos dados estão no site **OpenML**. Primeiro baixamos de lá e salvamos como CSV. Depois lemos o CSV com `pd.read_csv()`, que é o jeito como você vai abrir dados na maioria dos projetos.

In [44]:
from sklearn.datasets import fetch_openml

# baixa o dataset do OpenML e salva como arquivo CSV
fetch_openml(data_id=42688, as_frame=True).frame.to_csv("alugueis.csv", index=False)

In [45]:
# lê o arquivo CSV e guarda a tabela na variável df
df = pd.read_csv("alugueis.csv")

## 2.2 head() e tail()
`head()` mostra as 5 primeiras linhas e `tail()` as 5 últimas. Dentro dos parênteses dá para escolher quantas linhas, por exemplo `head(10)`.

In [46]:
df.head()

,city,area,rooms,bathroom,parking_spaces,floor,animal,furniture,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
0,Sao Paulo,70,2,1,1,7,acept,furnished,2065,3300,211,42,5618
1,Sao Paulo,320,4,4,0,20,acept,not furnished,1200,4960,1750,63,7973
2,Porto Alegre,80,1,1,1,6,acept,not furnished,1000,2800,0,41,3841
3,Porto Alegre,51,2,1,0,2,acept,not furnished,270,1112,22,17,1421
4,Sao Paulo,25,1,1,0,1,not acept,not furnished,0,800,25,11,836


In [47]:
df.tail()

,city,area,rooms,bathroom,parking_spaces,floor,animal,furniture,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
10687,Porto Alegre,63,2,1,1,5,not acept,furnished,402,1478,24,22,1926
10688,Sao Paulo,285,4,4,4,17,acept,not furnished,3100,15000,973,191,19260
10689,Rio de Janeiro,70,3,3,0,8,not acept,furnished,980,6000,332,78,7390
10690,Rio de Janeiro,120,2,2,2,8,acept,furnished,1585,12000,279,155,14020
10691,Sao Paulo,80,2,1,0,0,acept,not furnished,0,1400,165,22,1587


## 2.3 shape
`shape` mostra o tamanho da tabela no formato **(linhas, colunas)**.

In [48]:
df.shape

(10692, 13)

## 2.4 columns

In [49]:
df.columns

Index(['city', 'area', 'rooms', 'bathroom', 'parking_spaces', 'floor',
       'animal', 'furniture', 'hoa_(BRL)', 'rent_amount_(BRL)',
       'property_tax_(BRL)', 'fire_insurance_(BRL)', 'total_(BRL)'],
      dtype='object')

**O que significa cada coluna:**

| Coluna | Significado |
|---|---|
| `city` | cidade |
| `area` | área em m² |
| `rooms` | quartos |
| `bathroom` | banheiros |
| `parking_spaces` | vagas de garagem |
| `floor` | andar (0 = casa de um andar só) |
| `animal` | aceita animais? |
| `furniture` | mobiliado? |
| `hoa_(BRL)` | condomínio (Reais) |
| `rent_amount_(BRL)` | aluguel (Reais) |
| `property_tax_(BRL)` | IPTU (Reais) |
| `fire_insurance_(BRL)` | seguro incendio (Reais) |
| `total_(BRL)` | valor total (Reais) |

"BRL" é o código do real.

## 2.5 info()
`info()` mostra, para cada coluna: o nome, quantos valores preenchidos ela tem e o tipo de dado.

In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10692 entries, 0 to 10691
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   city                  10692 non-null  object
 1   area                  10692 non-null  int64 
 2   rooms                 10692 non-null  int64 
 3   bathroom              10692 non-null  int64 
 4   parking_spaces        10692 non-null  int64 
 5   floor                 10692 non-null  int64 
 6   animal                10692 non-null  object
 7   furniture             10692 non-null  object
 8   hoa_(BRL)             10692 non-null  int64 
 9   rent_amount_(BRL)     10692 non-null  int64 
 10  property_tax_(BRL)    10692 non-null  int64 
 11  fire_insurance_(BRL)  10692 non-null  int64 
 12  total_(BRL)           10692 non-null  int64 
dtypes: int64(10), object(3)
memory usage: 1.1+ MB


## 2.6 describe()
`describe()` calcula estatísticas das colunas numéricas: quantidade (`count`), média (`mean`), desvio padrão (`std`), mínimo (`min`), quartis (25%, 50%, 75%) e máximo (`max`).

In [51]:
df.describe()

,area,rooms,bathroom,parking_spaces,floor,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
count,10692.000000,10692.000000,10692.000000,10692.000000,10692.000000,1.069200e+04,10692.000000,10692.000000,10692.000000,1.069200e+04
mean,149.217920,2.506079,2.236813,1.609147,5.067995,1.174022e+03,3896.247194,366.704358,53.300879,5.490487e+03
std,537.016942,1.171266,1.407198,1.589521,6.069050,1.559231e+04,3408.545518,3107.832321,47.768031,1.648473e+04
min,11.000000,1.000000,1.000000,0.000000,0.000000,0.000000e+00,450.000000,0.000000,3.000000,4.990000e+02
25%,56.000000,2.000000,1.000000,0.000000,1.000000,1.700000e+02,1530.000000,38.000000,21.000000,2.061750e+03
50%,90.000000,2.000000,2.000000,1.000000,3.000000,5.600000e+02,2661.000000,125.000000,36.000000,3.581500e+03
75%,182.000000,3.000000,3.000000,2.000000,8.000000,1.237500e+03,5000.000000,375.000000,68.000000,6.768000e+03
max,46335.000000,13.000000,10.000000,12.000000,301.000000,1.117000e+06,45000.000000,313700.000000,677.000000,1.120000e+06


Para colunas de texto, `value_counts()` conta quantas vezes cada valor aparece:

In [52]:
df["city"].value_counts()

,count
city,
Sao Paulo,5887
Rio de Janeiro,1501
Belo Horizonte,1258
Porto Alegre,1193
Campinas,853


In [53]:
df["furniture"].value_counts()

,count
furniture,
not furnished,8086
furnished,2606


In [54]:
df["animal"].value_counts()

,count
animal,
acept,8316
not acept,2376


---
# 3. Seleção e Filtragem de Dados

## 3.1 Seleção de colunas
Uma coluna: colchetes com o nome dela. O resultado é uma **Series**.

In [55]:
df["city"]

,city
0,Sao Paulo
1,Sao Paulo
2,Porto Alegre
3,Porto Alegre
4,Sao Paulo
...,...
10687,Porto Alegre
10688,Sao Paulo
10689,Rio de Janeiro
10690,Rio de Janeiro


Várias colunas: uma **lista** de nomes dentro dos colchetes (por isso os colchetes duplos). O resultado é um **DataFrame**.

In [56]:
df[["city", "area", "rooms"]]

,city,area,rooms
0,Sao Paulo,70,2
1,Sao Paulo,320,4
2,Porto Alegre,80,1
3,Porto Alegre,51,2
4,Sao Paulo,25,1
...,...,...,...
10687,Porto Alegre,63,2
10688,Sao Paulo,285,4
10689,Rio de Janeiro,70,3
10690,Rio de Janeiro,120,2


## 3.2 Seleção de linhas
Para pegar linhas, usamos `loc` ou `iloc`.

## 3.3 loc e iloc
- **`loc`** seleciona pelo **rótulo** (o nome do índice e o nome da coluna).
- **`iloc`** seleciona pela **posição** (números, começando do 0), como se contasse as linhas e colunas.

O formato é sempre `[linhas, colunas]`.

In [57]:
# linha com índice 0, todas as colunas
df.loc[0]

,0
city,Sao Paulo
area,70
rooms,2
bathroom,1
parking_spaces,1
floor,7
animal,acept
furniture,furnished
hoa_(BRL),2065
rent_amount_(BRL),3300


In [58]:
# linhas 0 a 4 (no loc o final ENTRA), só algumas colunas
df.loc[0:4, ["city", "rent_amount_(BRL)"]]

,city,rent_amount_(BRL)
0,Sao Paulo,3300
1,Sao Paulo,4960
2,Porto Alegre,2800
3,Porto Alegre,1112
4,Sao Paulo,800


In [59]:
# 5 primeiras linhas e 3 primeiras colunas, por posição (no iloc o final NÃO entra)
df.iloc[0:5, 0:3]

,city,area,rooms
0,Sao Paulo,70,2
1,Sao Paulo,320,4
2,Porto Alegre,80,1
3,Porto Alegre,51,2
4,Sao Paulo,25,1


In [60]:
# última linha
df.iloc[-1]

,10691
city,Sao Paulo
area,80
rooms,2
bathroom,1
parking_spaces,0
floor,0
animal,acept
furniture,not furnished
hoa_(BRL),0
rent_amount_(BRL),1400


**Atenção:** no `loc`, `0:4` inclui a linha 4. No `iloc`, `0:5` vai até a posição 4 (o 5 não entra), igual às listas do Python.

## 3.4 Filtros condicionais
Um filtro seleciona só as linhas que obedecem a uma condição.

A parte de dentro, por exemplo `df["city"] == "Porto Alegre"`, cria uma lista de Verdadeiro/Falso, uma para cada linha. O `df[...]` de fora mantém só as linhas Verdadeiras.

In [61]:
df["city"] == "Porto Alegre"

,city
0,False
1,False
2,True
3,True
4,False
...,...
10687,True
10688,False
10689,False
10690,False


In [62]:
# só imóveis em Porto Alegre
df[df["city"] == "Porto Alegre"]

,city,area,rooms,bathroom,parking_spaces,floor,animal,furniture,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
2,Porto Alegre,80,1,1,1,6,acept,not furnished,1000,2800,0,41,3841
3,Porto Alegre,51,2,1,0,2,acept,not furnished,270,1112,22,17,1421
35,Porto Alegre,38,1,1,2,11,not acept,not furnished,450,1750,0,26,2226
39,Porto Alegre,40,1,1,1,6,acept,furnished,390,2990,0,44,3424
47,Porto Alegre,42,1,1,2,2,acept,furnished,190,1770,17,26,2003
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10645,Porto Alegre,400,4,2,2,15,acept,furnished,2250,6300,500,92,9142
10673,Porto Alegre,220,3,2,2,15,acept,not furnished,842,2400,117,36,3395
10676,Porto Alegre,40,1,1,0,1,acept,not furnished,330,1200,159,18,1707
10682,Porto Alegre,160,3,2,3,4,acept,furnished,850,3300,220,49,4419


In [63]:
# só imóveis com 3 quartos ou mais
df[df["rooms"] >= 3]

,city,area,rooms,bathroom,parking_spaces,floor,animal,furniture,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
1,Sao Paulo,320,4,4,0,20,acept,not furnished,1200,4960,1750,63,7973
5,Sao Paulo,376,3,3,7,0,acept,not furnished,0,8000,834,121,8955
7,Sao Paulo,213,4,4,4,4,acept,not furnished,2254,3223,1735,41,7253
15,Campinas,330,4,6,6,0,acept,furnished,680,8000,328,121,9129
19,Sao Paulo,600,4,5,6,0,acept,not furnished,0,12000,9500,181,21680
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10683,Sao Paulo,280,4,4,2,5,acept,not furnished,4200,4000,1042,51,9293
10685,Sao Paulo,83,3,2,2,11,acept,not furnished,888,7521,221,96,8726
10686,Sao Paulo,150,3,3,2,8,not acept,furnished,0,13500,0,172,13670
10688,Sao Paulo,285,4,4,4,17,acept,not furnished,3100,15000,973,191,19260


## 3.5 Múltiplas condições
Para juntar condições usamos `&` ("e") ou `|` ("ou"), e **cada condição vai entre parênteses**.

**Curiosidade:** no dataset, "aceita animais" está escrito `acept`, com um "c" só. Temos que digitar igual, senão o filtro não encontra nada.

In [64]:
# aluguel até R$ 2.000 E aceita animais
df[(df["rent_amount_(BRL)"] <= 2000) & (df["animal"] == "acept")]

,city,area,rooms,bathroom,parking_spaces,floor,animal,furniture,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
3,Porto Alegre,51,2,1,0,2,acept,not furnished,270,1112,22,17,1421
6,Rio de Janeiro,72,2,1,0,7,acept,not furnished,740,1900,85,25,2750
11,Campinas,46,1,1,1,10,acept,not furnished,550,580,43,8,1181
18,Rio de Janeiro,56,2,1,0,8,acept,not furnished,800,1220,0,16,2036
22,Sao Paulo,160,3,2,2,18,acept,furnished,1530,1900,167,25,3622
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10660,Sao Paulo,52,2,1,1,1,acept,not furnished,390,950,23,13,1376
10663,Sao Paulo,90,3,1,0,13,acept,not furnished,785,1600,0,21,2406
10668,Sao Paulo,24,1,1,0,0,acept,not furnished,0,870,0,14,884
10676,Porto Alegre,40,1,1,0,1,acept,not furnished,330,1200,159,18,1707


In [65]:
# imóveis em Campinas OU em Belo Horizonte
df[(df["city"] == "Campinas") | (df["city"] == "Belo Horizonte")]

,city,area,rooms,bathroom,parking_spaces,floor,animal,furniture,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
11,Campinas,46,1,1,1,10,acept,not furnished,550,580,43,8,1181
15,Campinas,330,4,6,6,0,acept,furnished,680,8000,328,121,9129
21,Belo Horizonte,42,1,1,1,17,not acept,furnished,470,2690,172,36,3368
27,Belo Horizonte,64,2,2,1,11,acept,not furnished,352,1500,80,20,1952
28,Campinas,208,3,2,4,0,acept,not furnished,0,3180,100,48,3328
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10656,Campinas,140,1,2,2,15,acept,not furnished,1462,5200,284,66,7012
10659,Campinas,150,3,2,4,0,acept,furnished,0,3500,186,53,3739
10661,Campinas,250,1,2,2,0,acept,not furnished,0,2200,602,34,2836
10665,Belo Horizonte,55,2,1,1,2,not acept,furnished,200,1600,75,22,1897


Filtro e `loc` podem ser usados juntos: o filtro escolhe as linhas e a lista escolhe as colunas.

In [66]:
df.loc[df["furniture"] == "furnished", ["city", "area", "rent_amount_(BRL)"]]

,city,area,rent_amount_(BRL)
0,Sao Paulo,70,3300
8,Sao Paulo,152,15000
9,Rio de Janeiro,35,2300
10,Sao Paulo,26,2100
13,Sao Paulo,55,4200
...,...,...,...
10682,Porto Alegre,160,3300
10686,Sao Paulo,150,13500
10687,Porto Alegre,63,1478
10689,Rio de Janeiro,70,6000


---
# 4. Manipulação de DataFrames

## 4.1 Renomeação de colunas
Deu para perceber que digitar `rent_amount_(BRL)` toda hora é chato e fácil de errar. Com `rename()` trocamos por nomes curtos em português.

In [67]:
df = df.rename(columns={
    "city": "cidade",
    "area": "area",
    "rooms": "quartos",
    "bathroom": "banheiros",
    "parking_spaces": "vagas",
    "floor": "andar",
    "animal": "animais",
    "furniture": "mobiliado",
    "hoa_(BRL)": "condominio",
    "rent_amount_(BRL)": "aluguel",
    "property_tax_(BRL)": "iptu",
    "fire_insurance_(BRL)": "seguro",
    "total_(BRL)": "total",
})
df.head()

,cidade,area,quartos,banheiros,vagas,andar,animais,mobiliado,condominio,aluguel,iptu,seguro,total
0,Sao Paulo,70,2,1,1,7,acept,furnished,2065,3300,211,42,5618
1,Sao Paulo,320,4,4,0,20,acept,not furnished,1200,4960,1750,63,7973
2,Porto Alegre,80,1,1,1,6,acept,not furnished,1000,2800,0,41,3841
3,Porto Alegre,51,2,1,0,2,acept,not furnished,270,1112,22,17,1421
4,Sao Paulo,25,1,1,0,1,not acept,not furnished,0,800,25,11,836


In [68]:
df["preco_m2"] = df["aluguel"] / df["area"]
df[["cidade", "area", "aluguel", "preco_m2"]].head()

,cidade,area,aluguel,preco_m2
0,Sao Paulo,70,3300,47.142857
1,Sao Paulo,320,4960,15.500000
2,Porto Alegre,80,2800,35.000000
3,Porto Alegre,51,1112,21.803922
4,Sao Paulo,25,800,32.000000


## 4.2 Criação de colunas
Para criar uma coluna, basta dar um nome novo e dizer como calcular. O Pandas faz a conta para todas as linhas de uma vez.

Um imóvel grande é naturalmente mais caro, então o **preço por m²** deixa a comparação entre imóveis mais justa.

In [69]:
df["preco_m2"] = df["aluguel"] / df["area"]
df[["cidade", "area", "aluguel", "preco_m2"]].head()

,cidade,area,aluguel,preco_m2
0,Sao Paulo,70,3300,47.142857
1,Sao Paulo,320,4960,15.500000
2,Porto Alegre,80,2800,35.000000
3,Porto Alegre,51,1112,21.803922
4,Sao Paulo,25,800,32.000000


## 4.3 Alteração de colunas
Também dá para mudar os valores de uma coluna que já existe. Aqui traduzimos os textos com `replace()`:

In [70]:
df["mobiliado"] = df["mobiliado"].replace({"furnished": "sim", "not furnished": "não"})
df["animais"] = df["animais"].replace({"acept": "sim", "not acept": "não"})
df[["mobiliado", "animais"]].head()

,mobiliado,animais
0,sim,sim
1,não,sim
2,não,sim
3,não,sim
4,não,não


E arredondar o preço por m² para 2 casas decimais:

In [71]:
df["preco_m2"] = df["preco_m2"].round(2)
df["preco_m2"].head()

,preco_m2
0,47.14
1,15.50
2,35.00
3,21.80
4,32.00


## 4.4 Remoção de linhas e colunas
`drop()` remove colunas (`columns=`) ou linhas (`index=`).

Removendo uma coluna que não vamos usar:

In [72]:
df = df.drop(columns=["seguro"])
df.columns

Index(['cidade', 'area', 'quartos', 'banheiros', 'vagas', 'andar', 'animais',
       'mobiliado', 'condominio', 'aluguel', 'iptu', 'total', 'preco_m2'],
      dtype='object')

Removendo linhas: primeiro encontramos o índice das linhas que queremos tirar, depois passamos para o `drop`. Exemplo: tirar imóveis sem banheiro.

In [73]:
sem_banheiro = df[df["banheiros"] == 0].index
df = df.drop(index=sem_banheiro)
df.shape

(10692, 13)

## 4.5 Ordenação de dados
`sort_values()` ordena a tabela por uma coluna. `ascending=False` ordena do maior para o menor.

In [74]:
# 5 aluguéis mais caros
df.sort_values("aluguel", ascending=False).head()

,cidade,area,quartos,banheiros,vagas,andar,animais,mobiliado,condominio,aluguel,iptu,total,preco_m2
2182,Sao Paulo,700,4,7,8,0,sim,não,0,45000,8750,54430,64.29
7748,Sao Paulo,350,3,3,3,0,sim,não,0,30000,560,31010,85.71
6947,Sao Paulo,486,8,4,6,0,sim,não,0,25000,2200,27580,51.44
2619,Sao Paulo,80,2,1,1,1,sim,não,875,24000,0,25180,300.00
5525,Sao Paulo,900,3,4,8,0,sim,não,0,20000,3813,24110,22.22


In [75]:
# 5 imóveis com menor preço por m²
df.sort_values("preco_m2").head()

,cidade,area,quartos,banheiros,vagas,andar,animais,mobiliado,condominio,aluguel,iptu,total,preco_m2
9241,Campinas,12732,3,2,0,3,sim,não,700,1600,96,2417,0.13
2397,Belo Horizonte,46335,4,8,5,11,sim,sim,960,8500,646,10220,0.18
5915,Sao Paulo,24606,5,4,4,12,sim,não,2254,8100,7859,18320,0.33
4114,Porto Alegre,900,5,5,2,4,sim,não,2700,1000,542,4257,1.11
4691,Sao Paulo,830,2,2,2,0,sim,não,0,2001,159,2191,2.41


## 4.6 Agrupamento com groupby()
`groupby()` separa a tabela em grupos (por exemplo, um por cidade) e calcula algo para cada grupo.

- `groupby("cidade")` → separa por cidade
- `["aluguel"].mean()` → calcula a média do aluguel em cada grupo
- `sort_values()` → ordena o resultado

In [76]:
df.groupby("cidade")["aluguel"].mean().sort_values()

,aluguel
cidade,
Porto Alegre,2337.699916
Campinas,2364.290739
Rio de Janeiro,3232.904064
Belo Horizonte,3664.127981
Sao Paulo,4652.793783


Aluguel médio por m² em cada cidade (comparação mais justa):

In [77]:
df.groupby("cidade")["preco_m2"].mean().sort_values()

,preco_m2
cidade,
Campinas,21.192907
Belo Horizonte,24.143712
Porto Alegre,25.575373
Rio de Janeiro,34.931113
Sao Paulo,38.389368


Imóvel mobiliado custa mais? E aceitar animais muda o preço?

In [78]:
df.groupby("mobiliado")["aluguel"].mean()

,aluguel
mobiliado,
não,3578.46092
sim,4882.28703


In [81]:
medias = df.groupby("mobiliado")["aluguel"].mean()
diferenca = medias["sim"] - medias["não"]
round(diferenca, 2)

np.float64(1303.83)

In [79]:
df.groupby("animais")["aluguel"].mean()

,aluguel
animais,
não,3464.212121
sim,4019.685786


In [82]:
medias = df.groupby("animais")["aluguel"].mean()
diferenca = medias["sim"] - medias["não"]
round(diferenca, 2)

np.float64(555.47)

Dá para calcular várias coisas de uma vez com `agg()`:

In [80]:
df.groupby("cidade")["aluguel"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
cidade,,,,
Belo Horizonte,1258,3664.127981,450,15000
Campinas,853,2364.290739,500,15000
Porto Alegre,1193,2337.699916,500,19000
Rio de Janeiro,1501,3232.904064,500,15000
Sao Paulo,5887,4652.793783,500,45000


---
# Conclusão
<!-- Albie e Mayara: completar depois de rodar o notebook -->

**O que descobrimos:**
- A cidade com o aluguel médio mais alto é São Paulo.
- No preço por m², a cidade mais cara é São Paulo.
- Imóveis mobiliados custam em média R$ 1.303,83 a mais que os não mobiliados.

**Resumo dos comandos:**

| Tema | Comandos |
|---|---|
| Estruturas | `pd.Series`, `pd.DataFrame`, `columns`, `index`, `dtypes` |
| Importação e exploração | `read_csv`, `head`, `tail`, `shape`, `info`, `describe`, `value_counts` |
| Seleção e filtragem | `df["col"]`, `df[["a", "b"]]`, `loc`, `iloc`, filtros com `&` e `|` |
| Manipulação | `rename`, criar coluna, `replace`, `round`, `drop`, `sort_values`, `groupby`, `agg` |